# Notebook 3: Parameter Sensitivity Analysis

Explores how the key model parameters — $\eta$ (temporary impact), $\gamma$ (permanent impact), $\sigma$ (volatility), and $\lambda$ (risk aversion) — affect:

1. The shape of the optimal trajectory
2. E[C] and Var[C]
3. The decay rate $\kappa$

Also includes heatmaps of E[C] as a function of $(\eta, \lambda)$ and $(\sigma, \lambda)$.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

from src.almgren_chriss import optimal_trajectory, cost_of_trajectory, twap_trajectory

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120

# Baseline parameters
X0, T0, N0       = 1_000_000, 1.0, 390
sigma0, gamma0   = 0.02, 2.5e-7
eta0, lam0       = 2.5e-6, 1e-6

## 1. Effect of σ on Optimal Trajectory

In [ ]:
sigmas = [0.005, 0.01, 0.02, 0.04, 0.08]
t_grid = np.linspace(0, T0, N0+1)

fig, ax = plt.subplots(figsize=(9, 4.5))
for sig in sigmas:
    t, h = optimal_trajectory(X0, T0, N0, sig, gamma0, eta0, lam0)
    kappa = np.sqrt(lam0 * sig**2 / eta0)
    ax.plot(t, h / 1e6, label=f'σ={sig:.3f}  (κ={kappa:.3f})', lw=1.8)

h_twap = twap_trajectory(X0, N0, T0)
ax.plot(t_grid, h_twap / 1e6, 'k--', lw=1.5, label='TWAP')
ax.set_xlabel('Time (days)')
ax.set_ylabel('Remaining shares (M)')
ax.set_title('Optimal Trajectory vs Volatility σ  (λ fixed)')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## 2. Effect of η on Optimal Trajectory

In [ ]:
etas = [5e-7, 1e-6, 2.5e-6, 5e-6, 1e-5]

fig, ax = plt.subplots(figsize=(9, 4.5))
for eta in etas:
    t, h = optimal_trajectory(X0, T0, N0, sigma0, gamma0, eta, lam0)
    kappa = np.sqrt(lam0 * sigma0**2 / eta)
    ax.plot(t, h / 1e6, label=f'η={eta:.1e}  (κ={kappa:.4f})', lw=1.8)

ax.plot(t_grid, h_twap / 1e6, 'k--', lw=1.5, label='TWAP')
ax.set_xlabel('Time (days)')
ax.set_ylabel('Remaining shares (M)')
ax.set_title('Optimal Trajectory vs Temporary Impact η  (λ, σ fixed)')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

print("Larger η = higher cost of fast trading → smoother trajectory (lower κ)")

## 3. Heatmap: E[C] as a Function of (η, λ)

In [ ]:
n_grid = 25
eta_vals = np.logspace(-7, -5, n_grid)    # 10x range around baseline
lam_vals = np.logspace(-8, -4, n_grid)

EC_grid = np.empty((n_grid, n_grid))

for i, eta in enumerate(eta_vals):
    for j, lam in enumerate(lam_vals):
        _, h = optimal_trajectory(X0, T0, 50, sigma0, gamma0, eta, lam)
        EC_grid[i, j] = cost_of_trajectory(h, T0, sigma0, gamma0, eta, X0)['expected_cost']

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.contourf(
    np.log10(lam_vals), np.log10(eta_vals),
    EC_grid / 1e6, levels=20, cmap='YlOrRd'
)
plt.colorbar(im, ax=ax, label='E[C] ($M)')
ax.set_xlabel(r'$\log_{10}(\lambda)$', fontsize=12)
ax.set_ylabel(r'$\log_{10}(\eta)$', fontsize=12)
ax.set_title(r'Heatmap: $E[C]$ (\$M) as a Function of $(\eta, \lambda)$', fontsize=13)

# Mark baseline
ax.scatter([np.log10(lam0)], [np.log10(eta0)], s=100, color='white',
           edgecolors='black', zorder=5, label='Baseline')
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('../data/fig_heatmap_eta_lambda.png', dpi=150)
plt.show()

## 4. Heatmap: Var[C] as a Function of (σ, λ)

In [ ]:
sigma_vals = np.linspace(0.005, 0.06, n_grid)
lam_vals2  = np.logspace(-8, -4, n_grid)

VAR_grid = np.empty((n_grid, n_grid))

for i, sig in enumerate(sigma_vals):
    for j, lam in enumerate(lam_vals2):
        _, h = optimal_trajectory(X0, T0, 50, sig, gamma0, eta0, lam)
        VAR_grid[i, j] = cost_of_trajectory(h, T0, sig, gamma0, eta0, X0)['variance']

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.contourf(
    np.log10(lam_vals2), sigma_vals,
    VAR_grid / 1e12, levels=20, cmap='Blues'
)
plt.colorbar(im, ax=ax, label=r'Var[C] ($\times 10^{12}$)')
ax.set_xlabel(r'$\log_{10}(\lambda)$', fontsize=12)
ax.set_ylabel(r'$\sigma$ (daily fractional vol)', fontsize=12)
ax.set_title(r'Heatmap: $\operatorname{Var}[C]$ as a Function of $(\sigma, \lambda)$', fontsize=13)
ax.scatter([np.log10(lam0)], [sigma0], s=100, color='white',
           edgecolors='black', zorder=5, label='Baseline')
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('../data/fig_heatmap_sigma_lambda.png', dpi=150)
plt.show()

## 5. κ as a Function of σ and η

In [ ]:
sig_grid = np.linspace(0.005, 0.06, 200)
eta_grid = np.logspace(-7, -5, 200)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

kappas_sig = np.sqrt(lam0 * sig_grid**2 / eta0)
axes[0].plot(sig_grid, kappas_sig * T0, color='steelblue', lw=2)
axes[0].axvline(sigma0, ls='--', color='grey', alpha=0.7)
axes[0].set_xlabel(r'$\sigma$', fontsize=12)
axes[0].set_ylabel(r'$\kappa T$', fontsize=12)
axes[0].set_title(r'$\kappa T$ vs $\sigma$  ($\eta, \lambda$ fixed)')

kappas_eta = np.sqrt(lam0 * sigma0**2 / eta_grid)
axes[1].semilogx(eta_grid, kappas_eta * T0, color='darkorange', lw=2)
axes[1].axvline(eta0, ls='--', color='grey', alpha=0.7)
axes[1].set_xlabel(r'$\eta$ (temporary impact)', fontsize=12)
axes[1].set_ylabel(r'$\kappa T$', fontsize=12)
axes[1].set_title(r'$\kappa T$ vs $\eta$  ($\sigma, \lambda$ fixed)')

plt.tight_layout()
plt.show()

print("κ increases with σ (timing risk penalty grows) and decreases with η (fast trading more costly).")

## 6. Cost Decomposition: Permanent vs Temporary

The permanent term $\frac{1}{2}\gamma X^2$ is independent of the schedule.  Only the temporary term varies.

In [ ]:
lambdas = np.logspace(-10, -3, 100)
permanent_cost = 0.5 * gamma0 * X0**2
temporary_costs = []

for lam in lambdas:
    _, h = optimal_trajectory(X0, T0, 50, sigma0, gamma0, eta0, lam)
    c = cost_of_trajectory(h, T0, sigma0, gamma0, eta0, X0)
    temporary_costs.append(c['expected_cost'] - permanent_cost)

temporary_costs = np.array(temporary_costs)

fig, ax = plt.subplots(figsize=(8, 4))
ax.semilogx(lambdas, temporary_costs / 1e3, label='Temporary impact cost', color='tomato', lw=2)
ax.axhline(0, color='grey', ls='--')
ax.set_xlabel(r'Risk aversion $\lambda$', fontsize=12)
ax.set_ylabel(r'$\eta \sum n_k^2/\tau$ (\$000)', fontsize=11)
ax.set_title('Temporary Impact Cost vs Risk Aversion\n(Permanent cost $\\frac{1}{2}\\gamma X^2$ is constant)', fontsize=12)
ax.legend()
plt.tight_layout()
plt.show()

print(f"Permanent cost (constant): ${permanent_cost:,.0f}")
print(f"Temporary cost range: ${temporary_costs.min():,.0f}  –  ${temporary_costs.max():,.0f}")